# Class 1 — LP Formulation with a Structured-Output Agent

**Goal:** turn a plain-English LP word problem into a Python object you can compute on.

**You'll learn, step by step:**

1. What a single LLM call actually is (it's just an HTTP request).
2. How a *system message* steers the model differently from a *user message*.
3. Why a string answer is useless for downstream code, and how *structured output* fixes that.
4. How the `Agent` abstraction in the OpenAI Agents SDK is just a wrapper over the same call.
5. Why the same prompt can give different answers (stochasticity), and how to reason about cost.

We'll use **Problem 1** from `LP_Problems_BUAD449.md` (four products, three machines, weekly profit) as the running example, then you'll formulate **Problem 2 or 3** yourself.

---

## 0. Setup

Before running anything below, make sure you have:

1. The project venv activated (it should be — Cursor uses `.venv` automatically).
2. A file `.env` at the **project root** (`BUAD449-AI-Module/.env`) containing one line:

   ```
   OPENAI_API_KEY=sk-...your-key...
   ```

   The `.env` file is git-ignored — your key never leaves your laptop.

Run the next cell. If it prints `OK`, you're good.

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# .env lives at the project root, one level up from this notebook.
PROJECT_ROOT = Path.cwd().parent
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — add it to .env at the project root"
client = OpenAI()
print("OK — client ready, model will be 'gpt-5.4-nano'")

MODEL = "gpt-5.4-nano"  # cheap model for demo

OK — client ready, model will be 'gpt-5.4-nano'


---
## 1. What a single LLM call actually is

Strip away every wrapper, and one LLM call is **one HTTP request**:

- **You send:** a list of messages (`role` + `content`) plus the model name.
- **You get back:** a response object containing the model's text plus token-usage info.

That's it. `Cursor`, `ChatGPT`, the `Agent` class — all of them are layered on top of this single primitive.

Run the cell below to make the most basic call possible.

In [11]:
response = client.responses.create(
    model=MODEL,
    input="In one sentence: what is a linear program?",
)

print("--- text the model returned ---")
print(response.output_text)
print()
print("--- token usage ---")
print(f"input tokens : {response.usage.input_tokens}")
print(f"output tokens: {response.usage.output_tokens}")
print(f"total tokens : {response.usage.total_tokens}")

--- text the model returned ---
A linear program is an optimization problem that seeks to maximize or minimize a linear objective function subject to linear equality/inequality constraints.

--- token usage ---
input tokens : 16
output tokens: 30
total tokens : 46


**What just happened:**

- We sent ~10 input tokens (your prompt, tokenized).
- The model produced ~30 output tokens (its sentence, tokenized).
- The whole call cost a fraction of a cent and took about a second.

Output tokens are roughly 4× more expensive than input tokens, so a verbose model answer is what really drives cost. Keep that in mind every time you design a prompt.

---
## 2. System message vs user message

When `input` is just a string, it becomes a single user-role message. To **steer** the model — set a persona, rules, output style — use a **system message**.

Think of it as the difference between:
- *user message* = the question you're asking, and
- *system message* = the standing instructions the model must follow when answering any question.

Watch how the same user question gets a different answer when we add a system message.

In [12]:
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": (
            "You are a strict operations-research professor. "
            "Answer in exactly two bullet points, no prose."
        )},
        {"role": "user", "content": "What is a linear program?"},
    ],
)
print(response.output_text)

- A **linear program (LP)** is an optimization model that maximizes or minimizes a **linear objective function** subject to **linear constraints** on decision variables, i.e., optimize \(c^T x\) subject to \(Ax \le b\) (and/or \(Ax=b\)) and variable sign restrictions (e.g., \(x\ge 0\)).  
- The feasible region formed by the linear constraints is a **polyhedron**, and optimal solutions (when they exist and are bounded) occur at an **extreme point (vertex)** of that polyhedron.


Notice: same question, very different output. The system message controls **format and behavior**; the user message controls **the request itself**. We will use the system message to lock the model into the role of "LP formulator" for the rest of this notebook.

---
## 3. Strings aren't programmatically usable

Here's our actual goal: take a word problem and produce something we can **feed into an LP solver** in Class 2.

Let's first try the naive way: just ask the model and see what comes back.

In [13]:
PROBLEM_1 = """
A certain plant can manufacture four products A, B, C, and D in any combination.
Each product requires time on each of three machines (in minutes per unit):

    Product   M1   M2   M3
    A         12    8    5
    B          7    9   10
    C          8    4    7
    D         10    0    3

Machine 1, 2, and 3 are available 20, 40, and 10 hours per week respectively.
Material costs are $2 each for products A and B and $1 each for C and D.
All products are competitive and any amounts made may be sold at $5, $4, $5, $5.
Variable labor costs are $4/hour for machines 1 and 2 and $3/hour for machine 3.
Find the best product mix to maximize total weekly profit.
"""

response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": "You are an LP formulation expert."},
        {"role": "user", "content": f"Formulate this LP problem:\n{PROBLEM_1}"},
    ],
)
answer = response.output_text
print(answer[:800], "...\n")

# The model gave us a beautifully written answer — but it's a string. Try to use it:
try:
    print("Objective expression:", answer.objective)  # noqa
except AttributeError as e:
    print("\nProblem:", e)
    print("We can READ this answer, but we can't programmatically pull out")
    print("'the objective' or 'the constraints' — it's just one big string.")

### Decision variables
Let
- \(x_A, x_B, x_C, x_D \ge 0\) = number of units of products \(A,B,C,D\) produced per week.

---

### Data (time in minutes per unit)
Machine availability per week:
- M1: 20 hours \(= 1200\) minutes
- M2: 40 hours \(= 2400\) minutes
- M3: 10 hours \(= 600\) minutes

Time coefficients (minutes/unit):
- \(A:\) M1 12, M2 8,  M3 5  
- \(B:\) M1 7,  M2 9,  M3 10  
- \(C:\) M1 8,  M2 4,  M3 7  
- \(D:\) M1 10, M2 0,  M3 3  

Material costs per unit:
- \(A,B:\$2\), \(C,D:\$1\)

Selling prices per unit:
- \(A:\$5,\; B:\$4,\; C:\$5,\; D:\$5\)

Variable labor costs per hour:
- M1: \$4/hour, M2: \$4/hour, M3: \$3/hour

Convert labor cost to **per minute**:
- M1: \(4/60 = 0.0666667\)
- M2: \(4/60 = 0.0666667\)
- M3: \(3/60 = 0.05\)

---

### Profit per unit (objective coeffi ...


Problem: 'str' object has no attribute 'objective'
We can READ this answer, but we can't programmatically pull out
'the objective' or 'the constraints' — it's just one big string.


We need the model's answer to come back as a **structured object** — one with named fields like `.objective`, `.constraints`, `.decision_variables`. That's what the next section is about.

---
## 4. Structured output with Pydantic

**Pydantic** lets you declare the *exact shape* of the answer you want, in regular Python class syntax. The OpenAI SDK then:

1. Translates your Pydantic class into a JSON schema.
2. Sends it to the model as part of the request.
3. Forces the model's output to match the schema.
4. Parses the JSON back into an instance of your class.

You write the schema once; you get a typed Python object back every time. Let's define one for an LP.

In [14]:
from typing import Literal
from pydantic import BaseModel, Field

class DecisionVariable(BaseModel):
    name: str = Field(description="Short symbol, e.g. 'X_A'")
    description: str = Field(description="What the variable represents in plain English")

class Objective(BaseModel):
    sense: Literal["maximize", "minimize"]
    expression: str = Field(
        description=(
            "Algebraic expression in the decision variables. "
            "Use Python operators: '*', '+', '-'. Convert all units first "
            "(e.g. minutes -> hours) so coefficients are concrete numbers."
        )
    )
    explanation: str = Field(description="Plain-English statement of what we are optimizing")

class Constraint(BaseModel):
    name: str = Field(description="Short snake_case label, e.g. 'machine_1_capacity'")
    expression: str = Field(
        description="Algebraic inequality, e.g. '12*X_A + 7*X_B <= 1200'"
    )
    explanation: str = Field(description="Plain-English meaning of the constraint")

class LPProblem(BaseModel):
    name: str = Field(description="Short title for the problem")
    summary: str = Field(description="One paragraph restating the problem in your own words")
    decision_variables: list[DecisionVariable]
    objective: Objective
    constraints: list[Constraint]

# Pydantic gives us auto-generated JSON schema for free:
import json
print(json.dumps(LPProblem.model_json_schema(), indent=2)[:600], "...")

{
  "$defs": {
    "Constraint": {
      "properties": {
        "name": {
          "description": "Short snake_case label, e.g. 'machine_1_capacity'",
          "title": "Name",
          "type": "string"
        },
        "expression": {
          "description": "Algebraic inequality, e.g. '12*X_A + 7*X_B <= 1200'",
          "title": "Expression",
          "type": "string"
        },
        "explanation": {
          "description": "Plain-English meaning of the constraint",
          "title": "Explanation",
          "type": "string"
        }
      },
      "required": [
        "name" ...


Now the call. Notice we use `responses.parse(...)` with `text_format=LPProblem`. The SDK takes care of the schema-translation and JSON-parsing for us.

In [15]:
FORMULATOR_INSTRUCTIONS = """
You are an expert at translating business word problems into linear programs.

Given a problem, return a complete LPProblem object that:
  - Defines one decision variable per choice the manager actually makes.
  - States the objective with all units already converted to a common base.
  - Includes every capacity, demand, and non-negativity constraint.
  - Uses concrete numeric coefficients (no symbolic constants).

Be terse. Be precise. The output will be consumed by a solver, not a human reader.
"""

parsed = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": FORMULATOR_INSTRUCTIONS},
        {"role": "user",   "content": PROBLEM_1},
    ],
    text_format=LPProblem,
)

lp: LPProblem = parsed.output_parsed
print(type(lp).__name__, "\u2014 a real Python object now")

LPProblem — a real Python object now


In [16]:
# Inspect it like any other Python object — that's the whole point.
print("Name      :", lp.name)
print("Sense     :", lp.objective.sense)
print("Objective :", lp.objective.expression)
print()
print("Decision variables:")
for v in lp.decision_variables:
    print(f"  {v.name:8s} = {v.description}")
print()
print("Constraints:")
for c in lp.constraints:
    print(f"  [{c.name}]  {c.expression}")

Name      : maximize_weekly_profit_product_mix_A_B_C_D
Sense     : maximize
Objective : (5-2-4*(12/60)-4*(8/60)-3*(5/60))*X_A + (4-2-4*(7/60)-4*(9/60)-3*(10/60))*X_B + (5-1-4*(8/60)-4*(4/60)-3*(7/60))*X_C + (5-1-4*(10/60)-4*(0/60)-3*(3/60))*X_D

Decision variables:
  X_A      = Weekly units of product A produced
  X_B      = Weekly units of product B produced
  X_C      = Weekly units of product C produced
  X_D      = Weekly units of product D produced

Constraints:
  [machine_1_capacity]  12*X_A + 7*X_B + 8*X_C + 10*X_D <= 1200
  [machine_2_capacity]  8*X_A + 9*X_B + 4*X_C + 0*X_D <= 2400
  [machine_3_capacity]  5*X_A + 10*X_B + 7*X_C + 3*X_D <= 600
  [nonneg_A]  X_A >= 0
  [nonneg_B]  X_B >= 0
  [nonneg_C]  X_C >= 0
  [nonneg_D]  X_D >= 0


**Compare** what you see above to the textbook formulation in `LP_Problems_BUAD449.md` (Problem 1):

- Did the model name the four decision variables correctly?
- Does its objective include both material costs AND labor costs?
- Did it convert minutes → hours? (If not, the constraint coefficients will be wrong.)
- Did it include three machine-capacity constraints plus non-negativity?

Try changing the system instructions above and re-running. This is **prompting in action** — and you can iterate in seconds because the call is just one cell.

---
## 5. The Agent abstraction — same call, less plumbing

Everything you just wrote (system message, user message, schema, parse) is what every LP-formulation call will look like. The `openai-agents` SDK packages that pattern into one object: an **`Agent`**.

An `Agent` is just:
- a name,
- standing instructions (the system message),
- an `output_type` (the Pydantic schema), and
- (later) a list of tools.

You call it with `await Runner.run(agent, user_input)`. Under the hood it builds the same `responses.parse(...)` request you just made by hand.

> **Why `await`?** Notebooks already run inside an asyncio event loop, so we use the async form `await Runner.run(...)`. In a standalone `.py` script you'd use either `Runner.run_sync(...)` or `asyncio.run(Runner.run(...))`. Same result, different plumbing.

In [17]:
from agents import Agent, Runner

formulator = Agent(
    name="LPFormulator",
    instructions=FORMULATOR_INSTRUCTIONS,
    model=MODEL,
    output_type=LPProblem,
)

# In a notebook, use `await Runner.run(...)` (top-level await works here).
# In a .py script, use `Runner.run_sync(...)` or `asyncio.run(Runner.run(...))`.
result = await Runner.run(formulator, PROBLEM_1)
lp_v2: LPProblem = result.final_output

print("Same shape as before:")
print("  sense     :", lp_v2.objective.sense)
print("  objective :", lp_v2.objective.expression)
print("  # vars    :", len(lp_v2.decision_variables))
print("  # constr  :", len(lp_v2.constraints))

Same shape as before:
  sense     : maximize
  objective : (5-2)*X_A + (4-2)*X_B + (5-1)*X_C + (5-1)*X_D - 4*((12/60)*X_A + (7/60)*X_B + (8/60)*X_C + (10/60)*X_D) - 4*((8/60)*X_A + (9/60)*X_B + (4/60)*X_C + (0/60)*X_D) - 3*((5/60)*X_A + (10/60)*X_B + (7/60)*X_C + (3/60)*X_D)
  # vars    : 4
  # constr  : 7


Same typed object, three lines of business logic instead of seven. The `Agent` abstraction starts to pay off in Classes 2 and 3, when we add tools and chain agents together.

---
## 6. Stochasticity: same prompt, different answers

LLMs sample from a probability distribution, so identical inputs can yield different outputs. For LP formulation, the **structure** should be stable (4 variables, 3 capacity constraints) but the *names* and *phrasings* can drift.

Run the next cell and look at how the count of variables/constraints behaves across three runs. If you ever see a different *count*, that's a real instability you'd want to harden the prompt against.

In [ ]:
for i in range(3):
    res = await Runner.run(formulator, PROBLEM_1)
    lp_i = res.final_output
    var_names = ", ".join(v.name for v in lp_i.decision_variables)
    print(f"Run {i+1}: {len(lp_i.decision_variables)} vars ({var_names}), "
          f"{len(lp_i.constraints)} constraints")

**How to deal with stochasticity in practice:**

- **Tighten the prompt.** Tell the model exactly what variables to use, e.g. "Always name product variables `X_A, X_B, ...`."
- **Validate the output.** A Pydantic schema only checks shape; you can also check semantics (e.g. "must have at least one capacity constraint per machine").
- **Re-prompt on failure.** If validation fails, send the error back to the model and ask it to fix.

We'll come back to this in Class 2 when the solver gives us a real, falsifiable signal: did the LP run, did it produce a sane number?

---
## 7. What does one formulation cost?

When we ran the formulator above, we got back a typed `LPProblem` — but also, quietly, a bill. The API **never tells you a dollar amount directly.** It only reports tokens:

```text
usage.input_tokens   # tokens we sent (system prompt + user problem + JSON schema)
usage.output_tokens  # tokens the model generated (the LPProblem JSON)
```

To turn tokens into money, you keep a **pricing table** on your side and multiply. Prices live on the [OpenAI pricing page](https://openai.com/api/pricing/) and change every few months — never hard-code them deep inside your application code.

Let's do the math for the Problem 1 run we did above with `formulator`.

In [ ]:
# --- tokens actually used by the Problem 1 run above ---
usage = result.context_wrapper.usage
print(f"input  tokens : {usage.input_tokens}")
print(f"output tokens : {usage.output_tokens}")
print(f"total  tokens : {usage.total_tokens}")

# --- pricing table (USD per 1,000,000 tokens) ---
# Look up current numbers: https://openai.com/api/pricing/
PRICES = {
    "gpt-5.4":      {"in": 2.5, "out": 15.00},  
    "gpt-5.4-nano": {"in": 0.2, "out":  1.25},  
}

def cost_usd(model: str, usage) -> float:
    p = PRICES[model]
    return usage.input_tokens / 1e6 * p["in"] + usage.output_tokens / 1e6 * p["out"]

one_call = cost_usd(MODEL, usage)
print(f"\none LP formulation with {MODEL}: ${one_call:.6f}")
print(f"  10   calls: ${one_call * 10:.4f}")
print(f"  1000 calls: ${one_call * 1_000:.2f}")
print(f"  1M   calls: ${one_call * 1_000_000:,.2f}")

**Two things to notice:**

1. **One call is cheap, but scale changes the conversation.** A single LP formulation costs less than a penny. A million of them — say, an enterprise planning tool running nightly across thousands of plants — is suddenly real money. Business students tend to under-estimate this gap; engineers tend to over-estimate it.
2. **Output tokens are ~8× more expensive than input tokens.** That's why we asked the model to be terse in `FORMULATOR_INSTRUCTIONS`. "Verbose = expensive" is a cost-engineering principle worth remembering.

In Homework 3, Part C, you'll re-run this exact calculation with a bigger model (`gpt-5.4`) and compare — the cost gap between models is often 10× or more for marginal quality gains.

---
# Homework 3 — Running LLM Agents via API

**Goal.** By the end of this homework you should be comfortable doing three things **on your own**, without copying a template:
1. Calling the OpenAI API with a system + user message.
2. Getting a typed answer back via a Pydantic `output_type`.
3. Wrapping the call in an `Agent` and running it with `await Runner.run(...)`.

## How to work on this homework

1. **Do NOT edit this notebook.** Keep it as a reference.
2. In the same folder (`part03-agent-with-structured-output/`), create a **new notebook** named:

   ```
   hw3_<your-last-name>.ipynb
   ```

3. In the **first cell** of your new notebook, copy the **setup and imports** you need from this notebook (the `load_dotenv` cell, the `LPProblem` schema cell, `FORMULATOR_INSTRUCTIONS`, and the `formulator` Agent). Re-defining everything in one place keeps your file self-contained — the grader shouldn't have to re-run the lecture notebook to check your work.
4. Do **Parts A, B, and C below** in that new file.
5. Execute every cell so outputs are visible when you submit.

> **Model for Parts A and B: use `gpt-5.4-nano`.** The setup cell already sets `MODEL = "gpt-5.4-nano"`, and `formulator` picks that up automatically — you do **not** need to pass the model argument explicitly when constructing the agent. Part C is where you deliberately switch to the bigger model to compare.

---

## Part A — Formulate three problems with `gpt-5.4-nano` (build familiarity)

Using the `formulator` agent (which uses **`gpt-5.4-nano`**), formulate **Problems 1, 2, and 4** from `LP_Problems_BUAD449.md`.

For each problem:

1. Create a Python string `PROBLEM_N = """..."""` containing the word problem only — drop the "Formulation" section of the markdown (that's the answer key, not input).
2. Call `result = await Runner.run(formulator, PROBLEM_N)` and assign `lp_N = result.final_output`.
3. Print `sense`, `objective.expression`, the number of decision variables, and the number of constraints.

You should end up with three `LPProblem` objects: `lp_1`, `lp_2`, `lp_4`.

---

## Part B — Evaluate against the answer key (build judgment)

Open `LP_Problems_BUAD449.md` and compare the model's output against the textbook formulation for each problem.

In a **markdown cell**, fill in this table:

| Problem | Right # of variables? | Right # of constraints? | Objective complete? | Units converted correctly? | Biggest mistake (one sentence) |
| ------- | --------------------- | ----------------------- | ------------------- | -------------------------- | ------------------------------ |
| 1       |                       |                         |                     |                            |                                |
| 2       |                       |                         |                     |                            |                                |
| 4       |                       |                         |                     |                            |                                |

---

## Part C — Model cost sensitivity (build cost intuition)

Bigger models are usually smarter but cost ~10× more per token. Part C makes you feel that trade-off with your own money.

Run **Problems 1 and 2** through **two** different models: **`gpt-5.4`** (the large model) and **`gpt-5.4-nano`** (the small model). That's four runs total — but the two `gpt-5.4-nano` runs are exactly the `lp_1` and `lp_2` you already produced in Part A, so you may reuse their usage data and only do two *new* runs with `gpt-5.4`.

### Step 1 — build two agents, one per model

```python
formulator_big = Agent(
    name="LPFormulator_big",
    instructions=FORMULATOR_INSTRUCTIONS,
    model="gpt-5.4",
    output_type=LPProblem,
)

formulator_nano = Agent(
    name="LPFormulator_nano",
    instructions=FORMULATOR_INSTRUCTIONS,
    model="gpt-5.4-nano",
    output_type=LPProblem,
)
```

### Step 2 — run and capture token usage

After `result = await Runner.run(agent, problem)`, you can read the token counts from:

```python
usage = result.context_wrapper.usage
usage.input_tokens     # tokens sent (prompt + schema)
usage.output_tokens    # tokens generated (the LPProblem JSON)
usage.total_tokens
```

Do this for each of the four (problem × model) combinations.

### Step 3 — compute approximate dollar cost

Look up the **current** per-million-token prices for both models on the [OpenAI pricing page](https://openai.com/api/pricing/) and plug them into a dict:

```python
PRICES = {
    "gpt-5.4":      {"in": _, "out": _},   # $ per 1M tokens — fill in
    "gpt-5.4-nano": {"in": _, "out": _},
}

def cost_usd(model, usage):
    p = PRICES[model]
    return usage.input_tokens / 1e6 * p["in"] + usage.output_tokens / 1e6 * p["out"]
```

### Step 4 — fill in this table (in a markdown cell)

| Problem | Model          | # vars | # constraints | Input tokens | Output tokens | Est. cost (USD) |
| ------- | -------------- | ------ | ------------- | ------------ | ------------- | --------------- |
| 1       | gpt-5.4        |        |               |              |               |                 |
| 1       | gpt-5.4-nano   |        |               |              |               |                 |
| 2       | gpt-5.4        |        |               |              |               |                 |
| 2       | gpt-5.4-nano   |        |               |              |               |                 |

### Step 5 — reflection (3–5 sentences in a markdown cell)

1. On which problem (1 or 2), if either, did the **nano** model give a noticeably worse formulation? How did you tell?
2. Roughly how many **times cheaper** is nano than the big model per run?
3. If a business deployed this agent to formulate LPs 10,000 times per day, which model would you choose, and why? There is no single right answer — explain your reasoning.

---

## Submission

- [ ] Parts A, B, and C completed in `hw3_<your-last-name>.ipynb`.
- [ ] Every cell executed so outputs are visible.
- [ ] Submit the **`.ipynb` file** to the **Homework 3** assignment on **Canvas**. Submit only the new file you created — do not submit this lecture notebook.

> **Cost tip.** Each run costs a few cents. Re-run only the one problem you're debugging, not all three.

---

### Looking ahead

- **Class 2 (LP solver).** We'll add a `solve_lp` *tool* the agent can call. Now the same `Agent` not only writes the LP — it actually solves it and returns numbers.
- **Class 3 (Spreadsheet).** We'll chain three agents into a controlled *workflow*: formulate → solve → write Excel report. You'll see the trade-off between letting the model decide steps (autonomy) and prescribing them (control).